In [1]:
import pickle, os
import numpy as np
from collections import Counter
from datasets import Dataset
from lowresource_llm_evaluation.LanguageDatasets import LanguageDataset
from lowresource_llm_evaluation.generateDataset import quitarYaAnotadas, generateInstructivoQA, generateInstructivoDataset
from transformers import AutoTokenizer
import pandas as pd
from dotenv import load_dotenv
load_dotenv("secrets.env")

MODELO = "Qwen/Qwen2.5-7B-Instruct" 
TOKENIZER = AutoTokenizer.from_pretrained(MODELO, trust_remote_code=True)

def load_custom_txt(file_path, ds_instance, name="mi_txt_chat"):
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    # Separamos por bloques, limpiamos espacios y filtramos vacíos
    # Usamos el delimitador de usuario para identificar cada "ejemplo"
    blocks = ["<|user|>" + b.strip() for b in content.split("<|user|>") if b.strip()]
    
    # Llamamos a tu método existente
    ds_instance.read_list(blocks, dataset_name=name)
    return ds_instance

# ---------------------------------------------------------
# 3. Función principal integrada
# ---------------------------------------------------------
def saveTrainTest(
    language,
    modelo_name, # Nombre para la carpeta
    tokenizer,
    thr=0.3,
    compute_length=True, # Por defecto True para auto-ajustar max_length
    concatenate=False,
    max_tokens=1024,
    N_max_lengths=20000,
    max_length=None
):
    root_dir = "TrainDatasets"
    os.makedirs(root_dir, exist_ok=True)

    # El nombre del modelo suele tener '/', lo limpiamos para la carpeta
    modelo_folder = modelo_name.split("/")[-1]
    save_dir = f"{root_dir}/{modelo_folder}"
    os.makedirs(save_dir, exist_ok=True)

    # 1. Carga y filtrado inicial
    ds = (
        LanguageDataset(language)
        .read_opus(source="NLLB", version=1)
        .filter_by_language(top_k=10, filter_language_thr=thr, batch_size=1024)
    )
    print("Estadísticas antes de concatenar")
    if compute_length:
        lengths = ds.get_stats(tokenizer, N_max=N_max_lengths)

    # 2. Concatenación (Nuevo método interno)
    if concatenate:
        ds.concatenate(tokenizer, max_tokens=max_tokens)
        if compute_length:
            print("Estadísticas después de concatenar")
            lengths = ds.get_stats(tokenizer, N_max=N_max_lengths)
    # 3. Estadísticas y Auto-ajuste de max_length
    # Si max_length es None, forzamos compute_length para saber el percentil 95
    if max_length is None and lengths is not None:
        max_length = int(np.percentile(lengths, 95))
        print(f"🎯 max_length ajustado automáticamente al P95: {max_length}")

    # 4. Split y Tokenización
    train = ds.tokenize(
        tokenizer=tokenizer,
        max_length=max_length
    )

    # 5. Guardar
    save_path = f"{save_dir}/{language}.pkl"
    with open(save_path, "wb") as f:
        pickle.dump(train, f, protocol=5)
    

    print(f"\nDataset guardado: {save_path} Train: {len(train)}")
    return train

def saveInstructivo(
    language,
    modelo_name, # Nombre para la carpeta
    tokenizer,
    compute_length=True, # Por defecto True para auto-ajustar max_length
    N_max_lengths=20000,
    max_length=None
):
    root_dir = "TrainDatasets"
    os.makedirs(root_dir, exist_ok=True)

    # El nombre del modelo suele tener '/', lo limpiamos para la carpeta
    modelo_folder = modelo_name.split("/")[-1]
    save_dir = f"{root_dir}/{modelo_folder}"
    os.makedirs(save_dir, exist_ok=True)

    dsInstructivo = LanguageDataset(language).read_local_file("TrainDatasets",f"Instructivo/{language}.csv").read_local_file("TrainDatasets",f"Instructivo_QA/{language}.csv")
    if os.path.exists(f"/notebooks/TrainDatasets/Generados-Instructivos/{language}.txt"):
        load_custom_txt(f"/notebooks/TrainDatasets/Generados-Instructivos/{language}.txt", dsInstructivo, "Instructivos-Generados-GPT")

    if compute_length or max_length is None:
        lengths = dsInstructivo.get_stats(tokenizer, N_max=N_max_lengths)
        
        if max_length is None and lengths is not None:
            max_length = int(np.percentile(lengths, 95))
            print(f"🎯 max_length ajustado automáticamente al P95: {max_length}")
        
        
    instructivoResult = dsInstructivo.tokenize(
            tokenizer=tokenizer,
            max_length=max_length
        )

    save_path = f"{save_dir}/{language}-Instructivo.pkl"
    with open(save_path, "wb") as f:
        pickle.dump(instructivoResult, f, protocol=5)
    print(f"\nDataset guardado: {save_path} Instructivo: {len(instructivoResult)}")
    return instructivoResult, dsInstructivo

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [ ]:
language = "asturiano"
saveTrainTest(
    language,
    MODELO,
    TOKENIZER,
    thr=0.5,
    compute_length=True,
    concatenate=False,
    max_tokens=160,
    N_max_lengths=200000, 
    max_length=120
)
saveInstructivo(
    language,
    MODELO, # Nombre para la carpeta
    TOKENIZER,
    compute_length=True, # Por defecto True para auto-ajustar max_length
    N_max_lengths=20000,
    max_length=160
)

Leyendo archivo Instructivo/asturiano.csv...
Leyendo archivo Instructivo_QA/asturiano.csv...

📊 Analizando estadísticas de tokens (N=1217)...
------------------------------------------------
Total líneas en dataset: 1217
Media: 209.05 | Mediana: 111.00
Percentil 95: 740.00 (recomendado para max_length)
Percentil 98: 905.76
Máximo: 1453 | Mínimo: 20
Moda (aprox): 40 tokens
------------------------------------------------



Map:   0%|          | 0/1217 [00:00<?, ? examples/s]


Dataset guardado: TrainDatasets/Qwen2.5-7B-Instruct/asturiano-Instructivo.pkl Instructivo: 1217


(Dataset({
     features: ['input_ids', 'attention_mask', 'labels'],
     num_rows: 1217
 }),
 <lowresource_llm_evaluation.LanguageDatasets.LanguageDataset at 0x7ff2461b2590>)

In [7]:
saveInstructivo(
    "aranes",
    MODELO, # Nombre para la carpeta
    TOKENIZER,
    compute_length=True, # Por defecto True para auto-ajustar max_length
    N_max_lengths=20000,
    max_length=160
)

Leyendo archivo Instructivo/aranes.csv...


Leyendo archivo Instructivo_QA/aranes.csv...

📊 Analizando estadísticas de tokens (N=1006)...
------------------------------------------------
Total líneas en dataset: 1006
Media: 287.15 | Mediana: 186.00
Percentil 95: 934.00 (recomendado para max_length)
Percentil 98: 1101.00
Máximo: 2013 | Mínimo: 26
Moda (aprox): 50 tokens
------------------------------------------------



Map:   0%|          | 0/1006 [00:00<?, ? examples/s]


Dataset guardado: TrainDatasets/Qwen2.5-7B-Instruct/aranes-Instructivo.pkl Instructivo: 1006


(Dataset({
     features: ['input_ids', 'attention_mask', 'labels'],
     num_rows: 1006
 }),
 <lowresource_llm_evaluation.LanguageDatasets.LanguageDataset at 0x7f9ccbbb5c90>)

In [3]:
language = "aranes"
saveTrainTest(
    language,
    MODELO,
    TOKENIZER,
    thr=0.5,
    compute_length=True,
    concatenate=False,
    max_tokens=160,
    N_max_lengths=200000, 
    max_length=120
)
saveInstructivo(
    language,
    MODELO, # Nombre para la carpeta
    TOKENIZER,
    compute_length=True, # Por defecto True para auto-ajustar max_length
    N_max_lengths=20000,
    max_length=160
)

[INFO] Descargando FastText LID-176 a /usr/local/lib/python3.11/dist-packages/lowresource_llm_evaluation/models/lid.176.ftz ...
[INFO] Modelo FastText descargado correctamente.


Empezando descarga de NLLB...
Procesando líneas directamente desde el flujo comprimido...
Dataset cargado: 29963156 líneas nuevas.
Estadísticas antes de concatenar

📊 Analizando estadísticas de tokens (N=200000)...
------------------------------------------------
Total líneas en dataset: 419116
Media: 36.61 | Mediana: 29.00
Percentil 95: 88.00 (recomendado para max_length)
Percentil 98: 111.00
Máximo: 193 | Mínimo: 5
Moda (aprox): 10 tokens
------------------------------------------------



Map:   0%|          | 0/419116 [00:00<?, ? examples/s]


Dataset guardado: TrainDatasets/Qwen2.5-7B-Instruct/aranes.pkl Train: 419116
Leyendo archivo Instructivo/aranes.csv...


Leyendo archivo Instructivo_QA/aranes.csv...

📊 Analizando estadísticas de tokens (N=1003)...
------------------------------------------------
Total líneas en dataset: 1003
Media: 287.87 | Mediana: 186.00
Percentil 95: 934.00 (recomendado para max_length)
Percentil 98: 1101.00
Máximo: 2013 | Mínimo: 26
Moda (aprox): 50 tokens
------------------------------------------------



Map:   0%|          | 0/1003 [00:00<?, ? examples/s]


Dataset guardado: TrainDatasets/Qwen2.5-7B-Instruct/aranes-Instructivo.pkl Instructivo: 1003


(Dataset({
     features: ['input_ids', 'attention_mask', 'labels'],
     num_rows: 1003
 }),
 <lowresource_llm_evaluation.LanguageDatasets.LanguageDataset at 0x7f5172dcb090>)

In [ ]:
import os
import pickle
import numpy as np
import urllib.request
import zipfile
import io
from pathlib import Path
from collections import deque

# =====================================================================
# CONFIGURACIÓN INICIAL
# =====================================================================
language = "gallego"
modelo_name = MODELO
tokenizer = TOKENIZER
thr = 0.3
compute_length = True
concatenate = False
max_tokens = 1024
N_max_lengths = 20000
max_length = 160

# Configuración del Corpus Nós
url_zenodo_zip = "https://zenodo.org/records/10687642/files/corpusnos.zip?download=1" 
zip_local_path = "corpus_nos.zip" # El ZIP es lo único que se mantendrá temporalmente

target_lines = 400000
min_len = 110
max_len = 180

# =====================================================================
# PROCESAMIENTO EN MEMORIA (ROUND-ROBIN)
# =====================================================================

estructura = {}

if not os.path.exists(zip_local_path):
    print("Descargando ZIP a disco (temporal)...")
    urllib.request.urlretrieve(url_zenodo_zip, zip_local_path)

print("Procesando ZIP directamente desde memoria...")
with zipfile.ZipFile(zip_local_path, 'r') as z:
    archivos_txt = [f for f in z.namelist() if f.endswith('.txt')]
    
    for file_path in archivos_txt:
        parts = Path(file_path).parts
        if len(parts) < 2: continue
        
        tema = parts[-2]
        nombre_archivo = parts[-1]
        
        if tema not in estructura:
            estructura[tema] = {}
        
        # CAMBIO AQUÍ: No usamos .read(), usamos el objeto archivo directamente
        with z.open(file_path) as f:
            # Envolvemos el archivo binario en un TextIOWrapper para leer línea a línea
            with io.TextIOWrapper(f, encoding='utf-8') as text_file:
                frases_validas = []
                for linea in text_file:
                    l_clean = linea.strip()
                    if min_len <= len(l_clean) <= max_len:
                        frases_validas.append(l_clean)
                
                if frases_validas:
                    np.random.shuffle(frases_validas)
                    estructura[tema][nombre_archivo] = deque(frases_validas)
        
        # Feedback visual para saber que no se ha colgado
        if len(estructura) % 10 == 0:
            print(f"Procesando tema: {tema}...", end="\r")

# Selección Round-Robin multinivel en memoria
frases_finales = []
temas_disponibles = list(estructura.keys())

print("Iniciando selección equilibrada en RAM...")
while len(frases_finales) < target_lines and temas_disponibles:
    for tema in list(temas_disponibles):
        archivos_en_tema = list(estructura[tema].keys())
        
        if not archivos_en_tema:
            temas_disponibles.remove(tema)
            continue
        
        for archivo in archivos_en_tema:
            if estructura[tema][archivo]:
                frases_finales.append(estructura[tema][archivo].popleft())
                if len(frases_finales) >= target_lines:
                    break
            else:
                del estructura[tema][archivo]
        
        if len(frases_finales) >= target_lines:
            break

# Limpiamos la estructura de la RAM para liberar espacio antes del pipeline
del estructura
print(f"Selección finalizada: {len(frases_finales)} frases listas en memoria.")

# =====================================================================
# PIPELINE DE DATASET (SIN PASAR POR TXT)
# =====================================================================

root_dir = "TrainDatasets"
modelo_folder = modelo_name.split("/")[-1]
save_dir = f"{root_dir}/{modelo_folder}"
os.makedirs(save_dir, exist_ok=True)

# Asumimos que LanguageDataset puede recibir una lista o lo inicializamos vacío
# Si tu clase LanguageDataset solo tiene read_lines, podemos "engañarla" con un buffer de memoria
ds = LanguageDataset(language)
ds.read_list(frases_finales, "corpus_nos")

# Aplicamos el resto del pipeline normalmente
ds.filter_by_language(top_k=10, filter_language_thr=thr, batch_size=1024)

if compute_length:
    lengths = ds.get_stats(tokenizer, N_max=N_max_lengths)

if max_length is None and lengths is not None:
    max_length = int(np.percentile(lengths, 95))
    print(f"🎯 max_length P95: {max_length}")

# Tokenización y creación del objeto final
train = ds.tokenize(tokenizer=tokenizer, max_length=max_length)

# EL ÚNICO GUARDADO EN DISCO: El archivo tokenizado final
save_path = f"{save_dir}/{language}.pkl"
with open(save_path, "wb") as f:
    pickle.dump(train, f, protocol=5)

# Borramos el ZIP al terminar para no dejar rastro de basura
if os.path.exists(zip_local_path):
    os.remove(zip_local_path)

print(f"\nProceso terminado. Solo se ha generado el archivo final: {save_path}")

Descargando ZIP a disco (temporal)...
Procesando ZIP directamente desde memoria...
Iniciando selección equilibrada en RAM...
Selección finalizada: 400000 frases listas en memoria.
[INFO] Descargando FastText LID-176 a /usr/local/lib/python3.11/dist-packages/lowresource_llm_evaluation/models/lid.176.ftz ...
[INFO] Modelo FastText descargado correctamente.



📊 Analizando estadísticas de tokens (N=20000)...
------------------------------------------------
Total líneas en dataset: 304997
Media: 40.60 | Mediana: 40.00
Percentil 95: 53.00 (recomendado para max_length)
Percentil 98: 56.00
Máximo: 77 | Mínimo: 10
Moda (aprox): 30 tokens
------------------------------------------------



Map:   0%|          | 0/304997 [00:00<?, ? examples/s]


Proceso terminado. Solo se ha generado el archivo final: TrainDatasets/Qwen2.5-7B-Instruct/gallego.pkl


In [3]:
import random
import time
import pandas as pd
from groq import Groq
from lowresource_llm_evaluation.LanguageDatasets import LanguageDataset

def quitarYaProcesados(dataset: LanguageDataset, anotadas: pd.DataFrame):
    """
    Filtra el dataset original eliminando los textos que ya tienen
    un instructivo/QA generado en el CSV.
    
    El CSV tiene columna "text" con formato:
    
    <|user|>
    INSTRUCCIÓN + TEXTO ORIGINAL
    <|assistant|>
    RESPUESTA
    """

    ya = set()

    # Extraer el texto original de cada ejemplo generado
    for full in anotadas["text"].astype(str):
        try:
            # El texto original está en la parte del <|user|>
            user_block = full.split("<|user|>")[1].split("<|assistant|>")[0].strip()

            # El original es SIEMPRE la última línea del bloque del usuario
            original = user_block.split("\n")[-1].strip()

            if original:
                ya.add(original)
        except:
            continue

    # Filtrar dataset original
    res = LanguageDataset(dataset.language)
    res.json = [entry for entry in dataset.json if entry["text"].strip() not in ya]

    return res


def safe_chat_completion_groq(client, model, messages, sleep_time=0.5, max_retries=5): 
    """ Llama a client.chat.completions.create con reintentos automáticos. Si falla (por ejemplo, error 429 o timeout), espera sleep_time y reintenta. """ 
    for attempt in range(max_retries): 
        try: 
            return client.chat.completions.create( model=model, messages=messages ) 
        except Exception as e: # Último intento → relanzar error 
            if attempt == max_retries - 1: 
                raise e # Espera antes del siguiente intento 
            wait = sleep_time * (attempt + 1) # backoff lineal 
            print(f"\nError en intento {attempt+1}: {e}. Reintentando en {wait} segundos...") 
            time.sleep(wait)

TEMPLATES = {
    "asturiano": [
        # ——— ALARGAR / AMPLIAR ———
        "Amplía esti testu añadiendo detalles, exemplos y esplicaciones:",
        "Desarrolla esti conteníu con una versión más llarga y completa:",
        "Esplica esti testu con más fondura y razonamientu:",
        "Amplía esti fragmentu como si fuera pa un adultu interesáu nel tema:",
        "Da una interpretación detallada y razonada d'esti conteníu:",
        "Crea una versión más estensa d'esti testu, añadiendo matices:",
        "Inventa un cuentu más llargu inspiráu nesti fragmentu:",
        "Crea un diálogu más desarrolláu basáu nesti conteníu:",
        "Da exemplos prácticos y amplía la información d'esti testu:",
        "Elabora una esplicación más completa d'esti conteníu:",

        # ——— VARIEDAD / REFORMULAR ———
        "Reformula esti testu con otres pallabres:",
        "Resume esti testu n'asturiano:",
        "Cambia esti testu a un tonu más formal:",
        "Cambia esti testu a un tonu más informal:"
    ],

    "gallego": [
        # ——— ALARGAR / AMPLIAR ———
        "Amplía este texto engadindo detalles, exemplos e explicacións:",
        "Desenvolve este contido cunha versión máis longa e completa:",
        "Explica este texto con máis profundidade e razoamento:",
        "Amplía este fragmento como se fose para un adulto interesado no tema:",
        "Dá unha interpretación detallada e razoada deste contido:",
        "Crea unha versión máis extensa deste texto, engadindo matices:",
        "Inventa un conto máis longo inspirado neste fragmento:",
        "Crea un diálogo máis desenvolvido baseado neste contido:",
        "Dá exemplos prácticos e amplía a información deste texto:",
        "Elabora unha explicación máis completa deste contido:",

        # ——— VARIEDAD / REFORMULAR ———
        "Reformula este texto con outras palabras:",
        "Resume este texto en galego:",
        "Cambia este texto a un ton máis formal:",
        "Cambia este texto a un ton máis informal:"
    ],

    "aranes": [
        # ——— ALARGAR / AMPLIAR ———
        "Amplie aguest tèxte en tot includir detalhs, exemples e explicacions:",
        "Desvolòpe aguest contengut damb ua version mès longa e completa:",
        "Explique aguest tèxte damb mès prigondor e rasonament:",
        "Amplie aguest fragment coma entà un adult interessat en eth tèma:",
        "Done ua interpretacion detalhada e rasonada d'aguest contengut:",
        "Cree ua version mès estensa d'aguest tèxte, includint matices:",
        "Invente un raconte mès long inspirat en aguest fragment:",
        "Cree un dialòg mès desvolopat basat en aguest contengut:",
        "Done exemples practics e amplie era informacion d'aguest tèxte:",
        "Elabòre ua explicacion mès completa d'aguest contengut:",

        # ——— VARIEDAD / REFORMULAR ———
        "Reformule aguest tèxte damb d'autes paraules:",
        "Resumís aguest tèxte en aranés:",
        "Càmbia aguest tèxte a un ton mès formau:",
        "Càmbia aguest tèxte a un ton mès informau:"
    ]
}


QA_TEMPLATES = {
    "asturiano": [
        "Inventa una pregunta razonable n'asturiano sobre esti testu y da una respuesta clara.",
        "Crea una pregunta útil basada nesti conteníu y da una respuesta completa.",
        "Xenera una pregunta de comprensión lectora y respóndela con detalle.",
        "Inventa una pregunta sencilla sobre esti fragmentu y da una respuesta curtia pero natural.",
        "Crea una pregunta abierta inspirada nesti testu y da una respuesta razonada."
    ],
    "gallego": [
        "Inventa unha pregunta razoable en galego sobre este texto e dá unha resposta clara.",
        "Crea unha pregunta útil baseada neste contido e ofrece unha resposta completa.",
        "Xera unha pregunta de comprensión lectora e respóndea con detalle.",
        "Inventa unha pregunta sinxela sobre este fragmento e dá unha resposta curta pero natural.",
        "Crea unha pregunta aberta inspirada neste texto e dá unha resposta razoada."
    ],
    "aranes": [
        "Invente ua question rasonabla en aranés sus aqueste tèxte e done ua responsa clara.",
        "Cree ua question util basada en aguest contengut e done ua responsa completa.",
        "Gènere ua question de compreneson deth tèxte e respòn en detalh.",
        "Invente ua question simpla sus aguest fragment e done ua responsa braca mès naturau.",
        "Cree ua question dubèrta inspirada en aguest tèxte e done ua responsa rasonada."
    ]
}

def generateInstructivoDataset(
        dataset,
        api_key,
        model="openai/gpt-oss-120b",
        N=2000,
        sleep_time=0.5,
        max_retries=5,
        save=False
    ):

    client = Groq(api_key=api_key)
    i= 0
    language = dataset.language
    if language not in TEMPLATES:
        raise ValueError(f"Idioma '{language}' non soportado.")

    templates = TEMPLATES[language]
    res_list = []

    base_texts = random.sample(dataset.json, N) if len(dataset.json) > N else dataset.json
    total = len(base_texts)
    i = 0

    for item in base_texts:
        callBegin = time.time()
        try:
            original = item["text"]
            template = random.choice(templates)
            instr = template + '\n' + original
            prompt_user = (
                f"{instr}\n\n"
                f"Devuelve SOLO en este formato EXACTO y respondiendo solo en {language}:\n"
                "<respuesta> ... </respuesta>"
            )
            # Llamada segura
            modified = safe_chat_completion_groq(client, sleep_time=sleep_time, max_retries=max_retries,
                model=model,
                messages=[
                    {"role": "user", "content": prompt_user}
                ]
            ).choices[0].message.content
            if modified is None:
                continue

            content = modified.strip()

            if "<respuesta>" not in content or "</respuesta>" not in content:
                continue

            
            resp  = content.split("<respuesta>")[1].split("</respuesta>")[0].strip()

            if len(instr) < 3 or len(resp) < 3:
                continue

            instruct_text = (
                "<|user|>\n" + instr + "\n"
                "<|assistant|>\n" + resp + "\n"
            )

            res_list.append(instruct_text)
            callEnd = time.time()

        except Exception as e:
            print(e)
            print("Guardando progreso actual")
            df = res_list
            return res_list

        # progreso
        i += 1
        pct = (i / total) * 100
        step = int(callEnd - callBegin)
        remaining = step * (total - i)
        print(
            f"\rProgreso: {pct:5.1f}% ({i}/{total}) | step: {step} s, "
            f"remaining time: {remaining//60} min {remaining%60} s",
            end=""
        )

    print(f"\nDataset instructivo generado ({language}): {len(res_list)} ejemplos válidos.")

    if save:
        df = pd.DataFrame({"text": res_list})
        date = time.localtime(time.time())
        filename = f"{dataset.language}_{model.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}"
        df.to_csv(filename, index=False)
        print(f"Guardado en {filename}")

    return res_list

def generateInstructivoQA(
        dataset,
        api_key,
        model="openai/gpt-oss-120b",
        N=2000,
        sleep_time=0.5,
        max_retries=5,
        save=False
    ):

    client = Groq(api_key=api_key)
    i= 0
    language = dataset.language
    if language not in QA_TEMPLATES:
        raise ValueError(f"Idioma '{language}' non soportado.")

    templates = QA_TEMPLATES[language]
    res_list = []

    base_texts = random.sample(dataset.json, N) if len(dataset.json) > N else dataset.json
    total = len(base_texts)
    i = 0

    for item in base_texts:
        callBegin = time.time()
        try:
            original = item["text"]
            template = random.choice(templates)

            prompt_user = (
                f"{template}\n\n"
                f"Texto base:\n{original}\n\n"
                f"Devuelve SOLO en este formato EXACTO, todo en {language}:\n"
                "<pregunta> GENERA SIEMPRE PREGUNTA </pregunta>\n"
                "<respuesta> GENERA SIEMPRE RESUPESTA </respuesta>"
            )

            modified = modified = safe_chat_completion_groq(client, sleep_time=sleep_time, max_retries=max_retries,
                model=model,
                messages=[
                    {"role": "user", "content": prompt_user}
                ]
            ).choices[0].message.content

            if modified is None:
                continue

            content = modified.strip()

            if "<pregunta>" not in content or "</pregunta>" not in content:
                continue
            if "<respuesta>" not in content or "</respuesta>" not in content:
                continue

            q = content.split("<pregunta>")[1].split("</pregunta>")[0].strip()
            a = content.split("<respuesta>")[1].split("</respuesta>")[0].strip()

            if len(q) < 3 or len(a) < 3:
                continue

            instruct_text = (
                "<|user|>\n" + q + "\n"
                "<|assistant|>\n" + a + "\n"
            )

            res_list.append(instruct_text)
            callEnd = time.time()

        except Exception as e:
            print(e)
            print("Guardando progreso actual")
            return res_list

         # progreso
        i += 1
        pct = (i / total) * 100
        step = int(callEnd - callBegin)
        remaining = step * (total - i)
        print(
            f"\rProgreso: {pct:5.1f}% ({i}/{total}) | step: {step} s, "
            f"remaining time: {remaining//60} min {remaining%60} s",
            end=""
        )

    print(f"\nDataset QA generado ({language}): {len(res_list)} ejemplos válidos.")

    if save:
        df = pd.DataFrame({"text": res_list})
        date = time.localtime(time.time())
        filename = f"{dataset.language}_{model.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}"
        df.to_csv(filename, index=False)
        print(f"Guardado en {filename}")

    return res_list


In [8]:
ast, oc, gl = (LanguageDataset(l, initializeTatoeba=True) for l in ["asturiano","aranes","gallego"])
for lang, dataset in [("asturiano", ast), ("aranes", oc), ("gallego", gl)]:
    print(f"\n=== Empezando {lang} ===")


    os.makedirs("TrainDatasets/Instructivo", exist_ok=True)
    os.makedirs("TrainDatasets/Instructivo_QA", exist_ok=True)
    out_instruct = f"TrainDatasets/Instructivo/{lang}.csv"
    out_qa       = f"TrainDatasets/Instructivo_QA/{lang}.csv"


    # Crear CSVs vacíos si no existen
    if not os.path.exists(out_instruct):
        print("Nuevo")
        pd.DataFrame(columns=["text"]).to_csv(out_instruct, index=False)
    if not os.path.exists(out_qa):
        pd.DataFrame(columns=["text"]).to_csv(out_qa, index=False)

    dataset.json = dataset[:500]
    
    # Cargar existentes
    df_inst_exist = pd.read_csv(out_instruct)
    df_qa_exist   = pd.read_csv(out_qa)
    print(df_inst_exist.shape, df_qa_exist.shape)
    
    # Filtrar dataset para no repetir
    dataset_inst = quitarYaProcesados(dataset, df_inst_exist)
    dataset_qa   = dataset
    dataset_qa.json = dataset_qa[df_qa_exist.shape[0]:]

    # Generar instructivos
    if df_inst_exist.shape[0] < 500:
        print(f"Generando instructivos para {lang}...")
        nuevos_inst = generateInstructivoDataset(
            dataset_inst,
            api_key=os.getenv("GROQ_API_KEY"),
            model="openai/gpt-oss-120b",
            save=False,
            N= 500 - df_inst_exist.shape[0]
        )

    if df_qa_exist.shape[0] < 500:
        # Generar QA
        print(f"Generando QA para {lang}...")
        nuevos_qa = generateInstructivoQA(
            dataset_qa,
            api_key=os.getenv("GROQ_API_KEY"),
            model="openai/gpt-oss-120b",
            save=False,
            N=500 - df_qa_exist.shape[0]
        )

    # Guardar concatenado
    if df_inst_exist.shape[0] < 500:
        df_final_inst = pd.concat([df_inst_exist, pd.DataFrame({"text": nuevos_inst})], ignore_index=True)
        df_final_inst.to_csv(out_instruct, index=False)
        print(f"✓ Nuevos instructivos: {len(nuevos_inst)}")
    if df_qa_exist.shape[0] < 500:
        df_final_qa = pd.concat([df_qa_exist, pd.DataFrame({"text": nuevos_qa})], ignore_index=True)
        df_final_qa.to_csv(out_qa, index=False)

        print(f"✓ Nuevos QA: {len(nuevos_qa)}")


print("\n=== Generación completada ===")


Descargando tatoeba para asturiano:
Completado con éxito
Descargando tatoeba para aranes:


Completado con éxito
Descargando tatoeba para gallego:


Completado con éxito

=== Empezando asturiano ===
(724, 1) (458, 1)
Generando QA para asturiano...

Dataset QA generado (asturiano): 0 ejemplos válidos.
✓ Nuevos QA: 0

=== Empezando aranes ===
(506, 1) (500, 1)

=== Empezando gallego ===


(326, 1) (5, 1)
Generando instructivos para gallego...
Progreso:  14.4% (25/174) | step: 4 s, remaining time: 9 min 56 ss
Error en intento 1: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01kdjbyd5aeftbs2nrxfpm9e93` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 6696, Requested 1418. Please try again in 855ms. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}. Reintentando en 0.5 segundos...
Progreso:  62.6% (109/174) | step: 3 s, remaining time: 3 min 15 ss
Error en intento 1: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01kdjbyd5aeftbs2nrxfpm9e93` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 6484, Requested 2205. Please try again in 5.1675s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/se